In [0]:
%sql
USE CATALOG ai_job_lakehouse;
USE SCHEMA gold;

-- 1. Dimension Table: Customers
CREATE OR REPLACE TABLE dim_customers AS
SELECT DISTINCT
    customerID AS customer_id,
    gender,
    CASE WHEN SeniorCitizen = 1 THEN 'Yes' ELSE 'No' END AS is_senior_citizen,
    Partner AS has_partner,
    Dependents AS has_dependents,
    tenure AS tenure_months,
    tenure_group
FROM ai_job_lakehouse.silver.telco_customers_clean;

-- 2. Dimension Table: Subscriptions & Services
CREATE OR REPLACE TABLE dim_services AS
SELECT DISTINCT
    customerID AS customer_id,
    Contract AS contract_type,
    PaperlessBilling AS is_paperless_billing,
    PaymentMethod AS payment_method,
    PhoneService AS phone_service,
    MultipleLines AS multiple_lines,
    InternetService AS internet_service,
    OnlineSecurity AS online_security,
    OnlineBackup AS online_backup,
    DeviceProtection AS device_protection,
    TechSupport AS tech_support,
    StreamingTV AS streaming_tv,
    StreamingMovies AS streaming_movies
FROM ai_job_lakehouse.silver.telco_customers_clean;

-- 3. Fact Table: Billing & Churn Metrics
CREATE OR REPLACE TABLE fact_churn_metrics AS
SELECT
    customerID AS customer_id,
    MonthlyCharges AS monthly_charges,
    TotalCharges AS total_charges,
    spend_tier,
    Churn AS churn_status,
    churn_flag,
    CASE WHEN churn_flag = 1 THEN MonthlyCharges ELSE 0.0 END AS monthly_revenue_lost
FROM ai_job_lakehouse.silver.telco_customers_clean;

-- 4. Verify Star Schema & Key Business KPIs
SELECT
    s.contract_type,
    COUNT(f.customer_id) AS total_customers,
    SUM(f.churn_flag) AS churned_customers,
    ROUND(AVG(f.churn_flag) * 100, 2) AS churn_rate_pct,
    ROUND(SUM(f.monthly_revenue_lost), 2) AS monthly_revenue_at_risk
FROM fact_churn_metrics f
JOIN dim_services s ON f.customer_id = s.customer_id
GROUP BY s.contract_type
ORDER BY churn_rate_pct DESC;

contract_type,total_customers,churned_customers,churn_rate_pct,monthly_revenue_at_risk
Month-to-month,3875,1655,42.71,120847.1
One year,1473,166,11.27,14118.45
Two year,1695,48,2.83,4165.3
